# IT 520 -- Which part to improve
## In-class activity: find the bottleneck in a kitchen

**20 minutes. Work with the person next to you.**

You are helping out at a restaurant. Tuesday nights, tickets that take six
minutes on a quiet evening are taking twelve during the rush, and the room is
unhappy.

The head chef has an answer already:

> "It's the oven. A tray takes six minutes and everything waits on it.
> Get me a second oven and this goes away."

A second oven costs money. Before anyone buys one, you are going to do what we
did with the server this morning: read the records the kitchen already keeps,
work out what one ticket costs each station, and find the tallest bar.

There are three stations. Every ticket gets **prepped**, some get **oven** time,
and every ticket gets **plated** at the pass.

**You are not timing anything.** Both records below were already written down.
Run the next two cells, then start at Step 1.


In [ ]:
# Run this cell. Nothing to edit here.
# It prints the two records the kitchen already keeps. You do not measure anything today.

TICKET_LOG = """
The point-of-sale ticket log  --  Tuesday 18:00-19:00 (the rush)

  ticket   fired   served   took
  0142     18:03   18:15    12 min    2 entrees, Concord booth
  0143     18:05   18:16    11 min    1 entree 1 salad, bar
  0144     18:07   18:20    13 min    2 entrees, table 9
     ... 12 more tickets between 18:00 and 19:00 ...

  tickets served: 15      average ticket: 12 min

Same log, a quiet Tuesday at 17:00, nobody waiting:

  tickets served: 5       average ticket: 6 min
"""

STATION_REPORT = """
The station busy-minutes report  --  same window, 18:00-19:00 (60 min)

  prep      busy 45 min     75%     chopping, portioning, sauces
  oven      busy 30 min     50%     5 trays, about 6 min a tray
  plating   busy 15 min     25%     the pass
"""

WINDOW_MIN   = 60   # minutes in the measured window
TICKETS      = 15   # tickets served in that window
QUIET_TICKET = 6    # minutes a ticket takes when nobody is waiting

print(TICKET_LOG)
print(STATION_REPORT)


In [ ]:
# Run this cell too. These are the checkers. Nothing to edit.

def bar(label, minutes, width=30, longest=3.0):
    n = max(1, min(width, int(round(minutes / longest * width))))
    return f"  {label:<10} {chr(9608)*n} {minutes:g} min per ticket"

def show_ledger(prep, oven, plating, title="What one ticket costs each station"):
    print(title)
    print()
    for lab, val in (("prep", prep), ("oven", oven), ("plating", plating)):
        print(bar(lab, val))
    print()

def check_ledger(prep, oven, plating):
    if None in (prep, oven, plating):
        print("STOP. Fill in all three numbers first."); return False
    show_ledger(prep, oven, plating)
    total = prep + oven + plating
    print(f"  Your ledger adds up to {total:g} minutes.")
    print(f"  The log says a ticket takes {QUIET_TICKET} minutes when nobody is waiting.")
    print()
    if abs(total - QUIET_TICKET) < 0.01:
        print("  PASS. Those match, so your three divisions are right. Go on.")
        return True
    print("  STOP. They do not match, so at least one division is wrong.")
    print("  Do not go on. Check each one: busy minutes for that station, divided by tickets served.")
    return False

def ceiling(minutes_per_ticket):
    return WINDOW_MIN / minutes_per_ticket

def report_ceiling(prep, oven, plating, label):
    tallest_name, tallest = max((('prep', prep), ('oven', oven), ('plating', plating)), key=lambda p: p[1])
    c = ceiling(tallest)
    print(label)
    print()
    show_ledger(prep, oven, plating, title="  the ledger now")
    print(f"  Tallest bar: {tallest_name}, {tallest:g} min per ticket.")
    print(f"  Ceiling: {WINDOW_MIN} min per hour / {tallest:g} min per ticket = {c:g} tickets an hour.")
    print(f"  Tonight the kitchen served {TICKETS} an hour.")
    print()
    return c

print("Checkers loaded.")


---
## Step 1 -- What does one ticket cost each station?

This is the same division we did for the disk this morning:

> **D = B / C** -- busy minutes of a part, divided by the jobs that finished.

Do it three times, once per station. Then run the cell.

The checker adds your three numbers up and compares them to the quiet-Tuesday
ticket time from the log. If they don't match, one of your divisions is wrong
and it will tell you so. **Don't go past Step 1 until it says PASS.**


In [ ]:
# STEP 1. Three divisions. One per station.
# Service demand D = busy minutes of that station / tickets served.
# Read the busy minutes off the station report above. TICKETS is already set for you.

prep_D    = None   # <-- replace None with your division, e.g.  45 / TICKETS
oven_D    = None   # <--
plating_D = None   # <--

check_ledger(prep_D, oven_D, plating_D)


---
## Step 2 -- The tallest bar, and the ceiling

Look at the bars you just printed.

1. **Which station is the tallest bar?** That is the bottleneck.
2. **What is the ceiling?** The station has 60 minutes an hour to give, and each
   ticket takes a fixed bite out of it. How many tickets an hour is the most it
   could possibly serve?

Before you fill it in: is it the station the chef named?


In [ ]:
# STEP 2. Name the bottleneck and work out the ceiling.

bottleneck = ""     # <-- "prep", "oven", or "plating"
ceiling_now = None  # <-- 60 minutes in the hour, divided by the tallest bar

def check_step2(name, c):
    tallest = max((('prep', prep_D), ('oven', oven_D), ('plating', plating_D)), key=lambda p: p[1])
    ok_name = (name.strip().lower() == tallest[0])
    ok_c = (c is not None and abs(c - WINDOW_MIN / tallest[1]) < 0.01)
    print("Bottleneck:", "correct" if ok_name else "not the tallest bar -- look at your ledger again")
    print("Ceiling:   ", "correct" if ok_c else "check it: 60 divided by the tallest bar")
    if ok_name and ok_c:
        print()
        print(f"  The {tallest[0]} station can serve at most {WINDOW_MIN/tallest[1]:g} tickets an hour.")
        print(f"  Tonight it served {TICKETS}. That is why the room backs up before it is full.")

check_step2(bottleneck, ceiling_now)


---
## Step 3 -- Now price the chef's answer

Buy the second oven. Run the cell and watch the ceiling.


In [ ]:
# STEP 3. The chef wants to buy a second oven. Change ONE number and run.
# A second oven halves what the oven costs each ticket.

oven_D_after = oven_D / 2

report_ceiling(prep_D, oven_D_after, plating_D, "AFTER BUYING A SECOND OVEN")


**Write down what happened to the ceiling.** Did it move? Why not?

The kitchen spent money and a quiet-night ticket got a little faster. The rush
did not.


---
## Step 4 -- The change that costs nothing

The vegetables are chopped to order during the rush. They don't have to be.
Chop them at four o'clock, when the kitchen is empty, and prep costs half as
much per ticket during service. Nothing is bought.

Run the cell.


In [ ]:
# STEP 4. The free change. Prep the vegetables before service instead of during it.
# That halves what prep costs each ticket. Nothing is bought.

prep_D_after = prep_D / 2

report_ceiling(prep_D_after, oven_D, plating_D, "AFTER MOVING PREP WORK OFF THE RUSH")


---
## Before you leave

Write these five lines on your sheet. Every one of them is a number you can
point at in a record.

```
What was slow:   ______________________________________________
Which station:   ____________ , ____ minutes of the ____ a ticket takes
The chef's fix:  a second oven moves the ceiling from ____ to ____
What I'd change: ______________________________________________
The new ceiling: ____ tickets an hour, and the tallest bar is now ____
```

**The one line to take with you:** the loudest station is not the bottleneck.
The tallest bar is. And only a change that shrinks the tallest bar moves the
ceiling.

*Submit a screenshot of this notebook, or your sheet, for today's participation.*
